In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from toponymy.annotation import NodeId, AnnotationTree, Annotation, AnnotationStore, Executor

14:41:53 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
14:41:53 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [3]:
import numpy as np

In [4]:
from toponymy.tools.notebook_data_load import load_small_newsgroups
newsgroups_df = load_small_newsgroups()
embeddings = np.stack(newsgroups_df["embedding"].values)
document_map = np.stack(newsgroups_df["map"].values)

In [5]:
from toponymy import ToponymyClusterer
clusterer = ToponymyClusterer(min_clusters=4, verbose=True)
clusterer.fit(document_map, embeddings);

Layer 0 found 8 clusters
Layer 1 found 4 clusters


In [6]:
clusterer.cluster_tree_

{(1, 0): [(0, 0)],
 (1, 1): [(0, 1)],
 (1, 2): [(0, 7), (0, 6), (0, 5)],
 (1, 3): [(0, 4)],
 (2, 0): [(1, 0), (1, 1), (1, 2), (1, 3), (0, 2), (0, 3)]}

In [7]:
node = NodeId(1, 0)

In [8]:
node.layer

1

In [9]:
node.cluster

0

In [10]:
node == (1, 0)

True

In [11]:
tree = AnnotationTree.from_clusterer(clusterer)

In [12]:
objects = newsgroups_df["post"].str.strip().values
embedding_vectors = embeddings
clusterable_vectors = document_map

## Exemplar Annotation


Need:
* centroid annotation
* cluster objects annotation
* cluster object vectors annotation

## AnnotationStore

In [13]:
store = AnnotationStore.from_clusterer(clusterer, objects, embedding_vectors)

In [14]:
executor = Executor(store)

## Compute exemplars

raw version

In [15]:
store

AnnotationStore(cluster_object_indices[12/12], cluster_objects[12/12], cluster_centroid[12/12], cluster_object_vectors[12/12])

In [16]:
layer = clusterer.cluster_layers_[0]

In [17]:
null_topic = np.mean(embedding_vectors, axis=0)

In [18]:
from toponymy.exemplar_texts import DiverseExemplarAnnotator

In [19]:
failures = executor.run(DiverseExemplarAnnotator(null_topic=null_topic,
                                                 n_exemplars=layer.n_exemplars,
                                                 diversify_alpha=layer.exemplars_diversify_alpha,
                                                 object_to_text_function=layer.object_to_text_function))

In [20]:
failures

{}

In [21]:
executor.store

AnnotationStore(cluster_object_indices[12/12], cluster_objects[12/12], cluster_centroid[12/12], cluster_object_vectors[12/12], exemplars[12/12], exemplar_original_indices[12/12])

In [22]:
cluster = store.node(NodeId(0, 0))
[f"{i}: {text[:30]}..." for i, text in zip(cluster["exemplar_original_indices"], cluster["exemplars"])]

['74: Well, it looks like, just as D...',
 '66: OK, you guys stirred up my chi...',
 '101: #21\tPETER AHOLA\t\tSeason: 2nd\nA...',
 '54: Is there any judgement call on...',
 '134: Tampa Bay                     ...',
 '111: that\nhis\npositive.\n\n\tClint Mal...',
 "115: Not if you've scored four runs...",
 '3: This just in...\n\nNolan Ryan hu...']

## Topic Annotator

WARNING: This requires a running LLM

In [23]:
from toponymy.prompt_construction import topic_name_prompt, topic_name_prompt_by_node

In [24]:
node = NodeId(0, 0)

In [25]:
node

NodeId(0, 0)

In [26]:
from toponymy.prompt_construction import SUMMARY_KINDS

In [27]:
def summary_kind(node, lowest_detail_level=0.0, highest_detail_level=1.0):
    detail_levels = np.linspace(
        lowest_detail_level,
        highest_detail_level,
        tree.n_layers
    )
    summary_level = int(round(detail_levels[node.layer] * (len(SUMMARY_KINDS) - 1)))
    summary_kind = SUMMARY_KINDS[summary_level]
    return summary_kind

In [28]:
prompt = topic_name_prompt_by_node(
        node=node,
        major_subtopics=[],
        minor_subtopics=[],
        other_subtopics=[],
        current_exemplars=store.exemplars[node],
        current_keyphrases=[],
        object_description="newsgroup posts",
        corpus_description="20-newsgroups dataset",
        summary_kind=summary_kind(node),
        exemplar_start_delimiter="<EXAMPLE_POST>\n",
        exemplar_end_delimiter="\n</EXAMPLE_POST>\n\n",
    )

In [29]:
from toponymy.templates import GET_TOPIC_NAME_REGEX
from toponymy.annotation import DescendantsInput

currently `TopicNameAnnotator` handles the subtopics computed from descendants in the tree but not yet the `make_subtopics` created from the bottom layer of topics. That will come, as the input/output specs to give the executor instructions needed to compute Annotators by layer at time still needs to be added. 

In [30]:
class TopicNameAnnotator:
    inputs = []
    optional_inputs = ["exemplars", "output_topics", "subtopics", "keyphrases"] # use it if it's there. don't if not.
    outputs = ["topics"]
    algorithm_input_type = {"exemplars": "node",
                      "output_topics": DescendantsInput(from_output="topics"),
                      "subtopics": "node",
                      "keyphrases": "node"}
    algorithm_output_type = "node"

    def __init__(self, llm_namer=None, prompt_template=None):
        self.llm=llm_namer
        self.prompt_template=prompt_template

    def annotate(
        self,
        node,
        *,
        exemplars=None,
        output_topics=None,
        subtopics=None,
        keyphrases=None,
        
    ):
        if exemplars is None: 
            exemplars = []
        if keyphrases is None:
            keyphrases = []

        if (output_topics is not None) and (len(output_topics) > 0): # slightly different logic here still
            if len(output_topics) == 1 and (list(output_topics.values())[0] != ""):
                print(f"{node}: returning")
                return {"topics" : list(output_topics.values())[0]}

            major_subtopics = [output_topic for child, output_topic in output_topics.items() if child.layer == node.layer - 1]
            minor_subtopics = [output_topic for child, output_topic in output_topics.items() if child.layer == node.layer - 2]

            # promote minor_subtopics to major_subtopics if there are not enough children one layer down    
            if len(major_subtopics) <= 1:
                major_subtopics = major_subtopics + minor_subtopics
                minor_subtopics = [output_topic for child, output_topic in output_topics.items() if child.layer < node.layer - 2]
    
            if node.layer > 1:
                other_subtopics = subtopics
            else:
                other_subtopics = []
            print(node)
            print(major_subtopics)
            print(minor_subtopics)
            print(other_subtopics)
        else:
            major_subtopics = []
            minor_subtopics = []
            other_subtopics = []
            

        prompt = topic_name_prompt_by_node(
                node=node,
                major_subtopics=major_subtopics,
                minor_subtopics=minor_subtopics,
                other_subtopics=other_subtopics,
                current_exemplars=exemplars,
                current_keyphrases=keyphrases,
                object_description="newsgroup posts",
                corpus_description="20-newsgroups dataset",
                summary_kind=summary_kind(node),
                exemplar_start_delimiter="<EXAMPLE_POST>\n",
                exemplar_end_delimiter="\n</EXAMPLE_POST>\n\n",
            )
        if isinstance(prompt, dict) or not prompt.startswith("[!SKIP!]: "):
            topic_name = self.llm.generate_topic_name(
                prompt,
                topic_extraction_function=(
                    self.prompt_template["layer"]["extract_topic_name"]
                    if self.prompt_template
                    else lambda json_response: str(json_response["topic_name"])
                ),
                get_topic_name_regex=(
                    self.prompt_template["layer"].get(
                        "get_topic_name_regex", GET_TOPIC_NAME_REGEX
                    )
                    if self.prompt_template
                    else GET_TOPIC_NAME_REGEX
                ),
            )
        else:
            topic_name = prompt.removeprefix("[!SKIP!]: ")
        if topic_name == '':
            raise ValueError("Failed to generate a valid topic name")
        return {"topics" : topic_name} # keys have to match outputs

In [31]:
from toponymy.llm_wrappers import OllamaNamer
from toponymy.debug_logging import BasicDebugLogger
from toponymy.tools.notebook_data_load import notebook_output_dir

debug_logger_file = notebook_output_dir() / "llm_debug_topic_annotation.jsonl"
debug_logger_file.write_text("", encoding="utf-8")  # Clear the file before starting
debug_logger = BasicDebugLogger(debug_logger_file, truncate=True)

namer = OllamaNamer(callback=debug_logger)
namer.test_llm_connectivity()  # Verify connection
namer.connectivity_status()

{'success': True,
 'model': 'ollama_chat/llama3.2',
 'wrapper': 'LiteLLMNamer',
 'response': '{"status": "ok"}',
 'error_type': None,
 'error_message': None,
 'original_exception': None}

In [32]:
executor.store

AnnotationStore(cluster_object_indices[12/12], cluster_objects[12/12], cluster_centroid[12/12], cluster_object_vectors[12/12], exemplars[12/12], exemplar_original_indices[12/12])

In [ ]:
failures = executor.run(TopicNameAnnotator(llm_namer=namer, prompt_template=layer.prompt_template))

In [34]:
failures

{NodeId(0, 0): 'ValueError: Failed to generate a valid topic name',
 NodeId(0, 3): 'ValueError: Failed to generate a valid topic name',
 NodeId(0, 6): 'ValueError: Failed to generate a valid topic name'}

In [35]:
store.topics.states

mappingproxy({NodeId(0, 0): <AnnotationState.FAILED: 'failed'>,
              NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 3): <AnnotationState.FAILED: 'failed'>,
              NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 6): <AnnotationState.FAILED: 'failed'>,
              NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>})

In [36]:
from toponymy.annotation import AnnotationState

In [37]:
for node in tree.nodes:
    if store.topics.states[node] == AnnotationState.COMPUTED:
        print(node, store.topics[node])
    else:
        print(node, store.topics.states[node])

(0, 0) AnnotationState.FAILED
(0, 1) Computer Hardware and Software Discussion
(0, 2) Cryptography
(0, 3) AnnotationState.FAILED
(0, 4) Car Maintenance and Safety Discussions
(0, 5) Criticism of Government and Corporate Practices
(0, 6) AnnotationState.FAILED
(0, 7) Christian Theology and Philosophy Discussions
(1, 0) Baseball
(1, 1) Computer Hardware and Software Discussion
(1, 2) Social Justice
(1, 3) Car Maintenance and Safety Discussions


In [38]:
from pathlib import Path
import json

In [39]:
def load_multiline_json_logs(path: str | Path) -> list[dict]:
    """
    Helper function to load a JSONL file.

    Args:
        path (str | Path): Path to the JSONL file.

    Returns:
        list[dict]: List of JSON objects loaded from the file.
    """
    path = Path(path)
    records = []

    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping bad JSON on line {i}: {e}")
    return records



In [40]:
debug_logs = load_multiline_json_logs(debug_logger_file)

In [41]:
for log in debug_logs:
    print(log["raw_response"])

{"baseball": "Sports Analysis and Discussion", "topic_specificity": 0.9}
{"Baseball Statistics and Analysis": 0.9, "topic_specificity": 0.9}
{"baseball": "baseball strategy and analysis", "topic_specificity": 0.9}
{"topic_name": "Computer Hardware and Software Discussion", "topic_specificity": 0.9}
{"cryptography-faq": "Public-Key Cryptography and Related Topics", "topic_specificity": 0.9}
{"topic_name": "Cryptography", "topic_specificity": 0.9}
{"Medical Discussion": 0.8, "Medical Discussion": 0.8}
{"Medical Discussion": 0.85, "topic_specificity": 0.85}
{"Medical Discussion": 0.83}
{"topic_name": "Car Maintenance and Safety Discussions", "topic_specificity": 0.9}
{"topic_name": "Criticism of Government and Corporate Practices", "topic_specificity": 0.83}
{"Middle East Conflict": 0.85}
{"Middle East Conflict": 0.85}
{"Middle East Conflict": 0.85}
{"Theology": 0.6, "Philosophy of Religion": 0.8, "Christianity": 0.9}
{"Religious Philosophy": 0.85, "specificity": 0.85}
{"topic_name": "Chr